In [66]:
# Pyomo and GLPK Installation
import pyomo.environ as pyo
pyo.SolverFactory('glpk').available()
# pyo.SolverFactory('glpk').executable() finds folder path

True

In [67]:
import matplotlib.pyplot as plt # Plot package
import matplotlib as mpl # Matplotlib package
import pandas as pd # Pandas package

In [68]:
from pyomo.environ import * # All pyomo environment files 
from pyomo.gdp import  * # All pyomo GDP files 

In [69]:
TASKS = {
    # Part 1
    ('Part_1','Mach_2'): {'dur':45,'prec':None}, # where dur(ation) is X minutes
    ('Part_1','Mach_3'): {'dur':10,'prec':('Part_1','Mach_2')}, # Dur(ation) is X minutes
    
    # Part 2
    ('Part_2','Mach_2'): {'dur':20,'prec':('Part_2','Mach_1')},
    ('Part_2','Mach_1'): {'dur':10,'prec':None}, 
    ('Part_2','Mach_3'): {'dur':34,'prec':('Part_2','Mach_2')},

    # Part 3...
    ('Part_3','Mach_2'): {'dur':12,'prec':('Part_3','Mach_3')}, 
    ('Part_3','Mach_1'): {'dur':17,'prec':('Part_3','Mach_2')},
    ('Part_3','Mach_3'): {'dur':28,'prec':None},
}

In [70]:
def PPS_model(TASKS):

  # initiate the model
  model = ConcreteModel()

  # Tasks
  model.TASKS = Set(initialize = TASKS.keys(), dimen = 2)
  # ('Part_1','Mach_2')

  # Jobs where j is job name, m is machine name
  model.JOBS = Set(initialize = list(set([j for (j,m) in model.TASKS])) )
  # ('Part_1')

  # Machines
  model.MACHINES = Set(initialize = list(set([m for (j,m) in model.TASKS])) )

  # Order of tasks ('Part_1','Mach_3'),('Part_1','Mach_2')
  # ('Part_1','Mach_3', 'Part_1','Mach_2')

  # * Cross-product
  model.TASKORDER = Set(initialize = model.TASKS * model.TASKS, dimen = 4,
                        filter = lambda model, j, m, k, n: (k,n) == TASKS[(j,m)]['prec'])

  # ('Part_1','Mach_2'),('Part_1','Mach_3') = True
  # TASKORDER = all pairs of tasks (j,m) and (k,n) such that (k,n) is the predecessor of (j,m).
  # dimen=4 because each element is a pair of tuples: (j,m,k,n).
  # filter ensures only the true predecessor relationships are kept.

  model.DISJUNCTIONS = Set(initialize = model.JOBS * model.JOBS * model.MACHINES, dimen = 3,
                           filter = lambda model, j, k, m: j != k and (j,m) in model.TASKS and (k,m) in model.TASKS)

  # DISJUNCTIONS defines all job pairs (j,k) that both need the same machine m.
  # For each such pair, we must enforce a disjunction ensuring they do NOT overlap.
  # j != k avoids pairing a job with itself.

  model.dur = Param(model.TASKS, initialize = lambda model, j, m: TASKS[(j,m)]['dur'])
  # Defines the duration of each task as a Pyomo parameter.

  ub = sum([model.dur[j,m] for (j,m) in model.TASKS])
  # Computes a simple upper bound (UB) = sum of all processing times.
  # Used as the max possible value for start times and makespan.

  # Decision variables
  model.makespan = Var(bounds = (0,ub))

  # Decision variable: total completion time of the entire schedule.
  model.start = Var(model.TASKS, bounds = (0,ub))

  # Decision variable: start time for each (job, machine) task.

   
  # Objectives
  model.objective = Objective(expr = model.makespan, sense = minimize)
  # Objective: minimize the makespan (standard goal for job-shop scheduling).

   
  # Define makespan
  # Finishing time for all jobs
  model.finish = Constraint(model.TASKS, rule = lambda model, j, m: 
                            model.start[j,m] + model.dur[j,m] <= model.makespan)

  # Ensures that every task finishes before or at the makespan.
  # Ensures that every task finishes before or at the makespan.


  # Processor
  model.preceding = Constraint(model.TASKORDER, rule = lambda model, j, m, k, n:
                               model.start[k,n] + model.dur[k,n] <= model.start[j,m])

  # Enforces precedence constraints.
  # If (k,n) is a predecessor of (j,m), then predecessor must finish BEFORE successor starts.


  # One machine - One part/job
  model.disjunctions = Disjunction(model.DISJUNCTIONS, rule = lambda model, j, k, m:
      [model.start[j,m] + model.dur[j,m] <= model.start[k,m],
       model.start[k,m] + model.dur[k,m] <= model.start[j,m]]
       )

  # Machine capacity constraint using disjunctions (GDP).
  # For any two jobs (j,k) on the same machine m:
  # One of the two conditions must be true:
  # - j finishes before k starts OR
  # - k finishes before j starts
  # This prevents overlap.


  TransformationFactory('gdp.hull').apply_to(model)
  # GDP model cannot be solved directly. 
  # 'hull' transformation converts disjunctions into mixed-integer linear constraints (MIP form).

  return model
  # Returns the fully constructed Pyomo model.


In [71]:
PPS_model(TASKS)

In [72]:
def PPS_model(model):
    SolverFactory('glpk').solve(model)
    results=[ 
        {'job': j,
        'Machine': m,
        'Start': model.start[j,m](),
        'Duration': model.dur[j,m],
        'Finish': model.start[(j,m)]() + model.dur[j,m]} for j,m in model.TASKS
    ]
    return results

In [73]:
def PPS(TASKS): 
    return PPS_solve(PPS_model(TASKS))

In [74]:
results = PPS(TASKS)
results

NameError: name 'PPS_solve' is not defined

In [65]:
def visualize(results):
    
    schedule = pd.DataFrame(results)
    JOBS = sorted(list(schedule['Job'].unique()))
    MACHINES = sorted(list(schedule['Machine'].unique()))
    makespan = schedule['Finish'].max()
    
    bar_style = {'alpha':1.0, 'lw':25, 'solid_capstyle':'butt'}
    text_style = {'color':'white', 'weight':'bold', 'ha':'center', 'va':'center'}
    colors = mpl.cm.Dark2.colors

    schedule.sort_values(by=['Job', 'Start'])
    schedule.set_index(['Job', 'Machine'], inplace=True)

    fig, ax = plt.subplots(2,1, figsize=(12, 5+(len(JOBS)+len(MACHINES))/4))

    for jdx, j in enumerate(JOBS, 1):
        for mdx, m in enumerate(MACHINES, 1):
            if (j,m) in schedule.index:
                xs = schedule.loc[(j,m), 'Start']
                xf = schedule.loc[(j,m), 'Finish']
                ax[0].plot([xs, xf], [jdx]*2, c=colors[mdx%7], **bar_style)
                ax[0].text((xs + xf)/2, jdx, m, **text_style)
                ax[1].plot([xs, xf], [mdx]*2, c=colors[jdx%7], **bar_style)
                ax[1].text((xs + xf)/2, mdx, j, **text_style)
                
    ax[0].set_title('Job Schedule')
    ax[0].set_ylabel('Job')
    ax[1].set_title('Machine Schedule')
    ax[1].set_ylabel('Machine')
    
    for idx, s in enumerate([JOBS, MACHINES]):
        ax[idx].set_ylim(0.5, len(s) + 0.5)
        ax[idx].set_yticks(range(1, 1 + len(s)))
        ax[idx].set_yticklabels(s)
        ax[idx].text(makespan, ax[idx].get_ylim()[0]-0.2, "{0:0.1f}".format(makespan), ha='center', va='top')
        ax[idx].plot([makespan]*2, ax[idx].get_ylim(), 'r--')
        ax[idx].set_xlabel('Time')
        ax[idx].grid(True)
        
    fig.tight_layout()
    plt.show()  

visualize(results)

NameError: name 'results' is not defined

In [82]:
# Simple robust optimization
Possible_durP1M2 = [32,31,35,90,55,78,89,45,65,67]

In [83]:
from statistics import mean
gamma = 0.8
durP1M2 = mean(Possible_durP1M2) + gamma * 0.5 * (max(Possible_durP1M2)-min(Possible_durP1M2))
durP1M2

82.30000000000001

In [86]:
Updated_TASKS = {
    # Part 1
    ('Part_1','Mach_2'): {'dur':durP1M2,'prec':None}, # where dur(ation) is X minutes
    ('Part_1','Mach_3'): {'dur':10,'prec':('Part_1','Mach_2')}, # Dur(ation) is X minutes
    
    # Part 2
    ('Part_2','Mach_2'): {'dur':20,'prec':('Part_2','Mach_1')},
    ('Part_2','Mach_1'): {'dur':10,'prec':None}, 
    ('Part_2','Mach_3'): {'dur':34,'prec':('Part_2','Mach_2')},

    # Part 3...
    ('Part_3','Mach_2'): {'dur':12,'prec':('Part_3','Mach_3')}, 
    ('Part_3','Mach_1'): {'dur':17,'prec':('Part_3','Mach_2')},
    ('Part_3','Mach_3'): {'dur':28,'prec':None},
}

In [89]:
def PPS(Updated_TASKS):
    return PPS_solve(PPS_model(Updated_TASKS))

results2 = PPS(Updated_TASKS)
visualize(results2)

NameError: name 'PPS_solve' is not defined